In [ ]:
!pip install osfclient


In [ ]:
import os
import urllib.request
import zipfile
import shutil
from google.colab import drive, userdata

# ==========================================
# 1. MOUNT GOOGLE DRIVE & SETUP API
# ==========================================
print("\n[SECTION 1] Initializing MLOps Environment...")
# mount Drive to bypass Colab storage and for persist datasets across Architectures
print("-> Mounting Google Drive")
drive.mount('/content/drive')

print("-> Creating permanent dataset directories in Google Drive...")
drive_dir = "/content/drive/MyDrive/NST_Datasets"

# Organises into Pytorch imagefolder compatible structure (class_1 subfolders)
coco_target = os.path.join(drive_dir, "coco_subset/images/class_1")
npr_target = os.path.join(drive_dir, "npr_subset/images/class_1")

os.makedirs(coco_target, exist_ok=True)
os.makedirs(npr_target, exist_ok=True)
print("-> Directory structure verified.")

# ==========================================
# 2. DOWNLOAD & EXTRACT COCO-STUFF 10K
# ==========================================
print("\n[SECTION 2] Processing COCO-Stuff 10k Dataset...")
coco_zip_path = "/content/cocostuff-10k.zip"
coco_url = "http://calvin.inf.ed.ac.uk/wp-content/uploads/data/cocostuffdataset/cocostuff-10k-v1.1.zip"

print("-> Downloading COCO-Stuff 10k..")
# Chrome User-Agent prevent host server from blocking urllib req
req = urllib.request.Request(
    coco_url,
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
)

# Open the URL and stream it directly to the Colab disk
with urllib.request.urlopen(req) as response, open(coco_zip_path, 'wb') as out_file:
    # Stream extraction buffer I/O to prevent OOM
    shutil.copyfileobj(response, out_file)

print("-> Extracting COCO-Stuff 10k images directly to Google Drive...")
with zipfile.ZipFile(coco_zip_path, 'r') as zip_ref:
    for member in zip_ref.infolist():
        # Only extract JPG images inside the 'images/' folder no extra metafolders
        if member.filename.startswith("images/") and member.filename.lower().endswith(".jpg"):
            filename = os.path.basename(member.filename)
            if filename:
                source = zip_ref.open(member)
                #reads from local zip and writes directly to drive
                target = open(os.path.join(coco_target, filename), "wb")
                with source, target:
                    shutil.copyfileobj(source, target)

# Delets zip archive from VM to free space
print("-> Cleaning up COCO-Stuff temporary zip file...")
if os.path.exists(coco_zip_path):
    os.remove(coco_zip_path)

print(f"-> COCO-Stuff 10k Complete! ({len(os.listdir(coco_target))} images currently saved).")

# ==========================================
# 3. FINAL VERIFICATION
# ==========================================
print("\n[SECTION 4] Verification...")
print("==========================================")
print("       DOWNLOAD & TRANSFER COMPLETE!      ")
print("==========================================")
print(f"Total Geometric Prior Images (COCO-Stuff 10k): {len(os.listdir(coco_target))}")
print(f"Total Stylistic Prior Images (Artworks):       {len(os.listdir(npr_target))}")
print("==========================================")

In [ ]:
# ==========================================
# 1. SETUP GOOGLE DRIVE TARGET
# ==========================================
# force_remount clear stale mounts states from causing colab I/O hangs
drive.mount('/content/drive', force_remount=True)
TARGET_DIR = "/content/drive/MyDrive/NST_Datasets/style_subset/images/style_targets"
os.makedirs(TARGET_DIR, exist_ok=True)

# ==========================================
# 2. THE CANONICAL NST STYLE TARGETS
# ==========================================
# source style targets from Johnson's github, as used for benchmark
STYLE_URLS = {
    "starry_night.jpg": "https://raw.githubusercontent.com/jcjohnson/fast-neural-style/master/images/styles/starry_night.jpg",
    "the_scream.jpg": "https://raw.githubusercontent.com/jcjohnson/fast-neural-style/master/images/styles/the_scream.jpg",
    "mosaic.jpg": "https://raw.githubusercontent.com/jcjohnson/fast-neural-style/master/images/styles/mosaic.jpg",
    "candy.jpg": "https://raw.githubusercontent.com/jcjohnson/fast-neural-style/master/images/styles/candy.jpg",
    "feathers.jpg": "https://raw.githubusercontent.com/jcjohnson/fast-neural-style/master/images/styles/feathers.jpg",
    "composition_vii.jpg": "https://raw.githubusercontent.com/jcjohnson/fast-neural-style/master/images/styles/composition_vii.jpg",
    "la_muse.jpg": "https://raw.githubusercontent.com/jcjohnson/fast-neural-style/master/images/styles/la_muse.jpg",
    "udnie.jpg": "https://raw.githubusercontent.com/jcjohnson/fast-neural-style/master/images/styles/udnie.jpg",
    "wave.jpg": "https://raw.githubusercontent.com/jcjohnson/fast-neural-style/master/images/styles/wave.jpg"
}

# ==========================================
# 3. DIRECT-TO-GDRIVE INGESTION
# ==========================================
print(f"-> Ingesting style priors into: {TARGET_DIR}")

downloaded = 0
for filename, url in STYLE_URLS.items():
    dest_path = os.path.join(TARGET_DIR, filename)

    # Skip if already downloaded it to save bandwidth
    if not os.path.exists(dest_path):
        print(f"   Downloading: {filename} ...")
        # injects a user agent to bypass server bot blocking
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(dest_path, 'wb') as out_file:
            out_file.write(response.read())
        downloaded += 1
    else:
        print(f"   Already exists (Skipping): {filename}")

print(f"\n-> SUCCESS: {downloaded} new style targets secured.")